# HIPPIE Cross-Dataset Training and Evaluation

This notebook implements a **cross-dataset transfer learning pipeline** for neuron classification using the HIPPIE multimodal CVAE framework.

## Overview

The pipeline consists of 4 phases:

1. **Pretraining**: Train on multiple datasets (excluding training/predict datasets)
2. **Fine-tuning**: Adapt to the training dataset without labels (unsupervised)
3. **Supervised Training**: Train with labels on the training dataset
4. **Evaluation**: Extract embeddings and evaluate cross-dataset performance using KNN

## Key Features

- **Trimodal learning**: Waveforms, ISI distributions, and autocorrelograms
- **Multiple configurations**: From baseline to full model with augmentations
- **Data augmentation**: Light, heavy, and ablation modes
- **Class balancing**: Optional weighted sampling for imbalanced datasets
- **Robust evaluation**: Cross-validation for k selection, balanced accuracy, confusion matrices

## 1. Setup and Imports

In [94]:
import sys
import os
import time
import warnings
warnings.filterwarnings('ignore')

# Add hippie module to path
code_dir = os.path.abspath(os.path.join(os.getcwd(), 'hippie'))
sys.path.append(code_dir)

# Core imports
import torch
import torch.nn as nn
import torch.optim as optim
import pytorch_lightning as pl
from pytorch_lightning.callbacks import Timer
import pandas as pd
import numpy as np
import wandb

# Scikit-learn imports
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import balanced_accuracy_score, confusion_matrix
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import random_split, WeightedRandomSampler

# HIPPIE imports
from dataloading import MultiModalEphysDataset, EphysDatasetLabeled, none_safe_collate
from multimodal_model import MultiModalCVAE, MultiModalCVAETrainModule, CVAEConfig, ExperimentConfigs
from utils import make_confmat, get_embeddings
from augmentations import AugmentedMultiModalEphysDataset

# Check for psutil
try:
    import psutil
    _HAS_PSUTIL = True
except Exception:
    _HAS_PSUTIL = False
    print("Warning: psutil not available, CPU memory monitoring will be disabled")

print("All imports successful")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

All imports successful
PyTorch version: 2.8.0
CUDA available: False


## 2. Configuration

Configure your experiment here. All parameters are explained in detail.

In [95]:
# Dataset to train on (must have labels)
TRAINING_DATASET = "hausser_cell_type"

# Dataset to predict on (will evaluate cross-dataset generalization)
PREDICT_DATASET = "mouse_organoids_cell_line"#mouse_organoids_cell_line lissberger_labeled_cell_type

# Available datasets with source IDs, you can add all the datasets you want here
AVAILABLE_DATASETS = {
    "hausser_cell_type": 1,
    "hull_cell_type": 2,
    "lissberger_labeled_cell_type": 3,
    "mouse_organoids_cell_line": 4
}

print(f"Training on: {TRAINING_DATASET}")
print(f"Predicting on: {PREDICT_DATASET}")
print(f"Available datasets: {list(AVAILABLE_DATASETS.keys())}")

Training on: hausser_cell_type
Predicting on: mouse_organoids_cell_line
Available datasets: ['hausser_cell_type', 'hull_cell_type', 'lissberger_labeled_cell_type', 'mouse_organoids_cell_line']


In [96]:
# Model configuration
MODEL_CONFIG = "augmentation_ablation"
Z_DIM = 20
BETA = 0.9

# Modality weights
WAVE_WEIGHT = 1.0
ISI_WEIGHT = 1.0
ACG_WEIGHT = 1.0

print(f"Configuration: {MODEL_CONFIG}")
print(f"Latent dimension: {Z_DIM}")
print(f"Beta: {BETA}")

Configuration: augmentation_ablation
Latent dimension: 20
Beta: 0.9


In [97]:
# Training parameters
LEARNING_RATE = 0.001
WEIGHT_DECAY = 0.01
BATCH_SIZE = 512
SUPERVISED_BATCH_SIZE = 64
GRADIENT_CLIP_VAL = 1.0

# Epoch limits
PRETRAIN_MAX_EPOCHS = 1#100
FINETUNE_MAX_EPOCHS = 1#10
SUPERVISED_MAX_EPOCHS = 1#10
EARLY_STOPPING_PATIENCE = 30

# Data splits
TRAIN_VAL_SPLIT = 0.8
FINETUNE_SPLIT = 0.2

# Other options
FINETUNE_WITHOUT_LABELS = True
USE_BALANCED_SAMPLING = True

print(f"Learning rate: {LEARNING_RATE}")
print(f"Max epochs: pretrain={PRETRAIN_MAX_EPOCHS}, finetune={FINETUNE_MAX_EPOCHS}, supervised={SUPERVISED_MAX_EPOCHS}")

Learning rate: 0.001
Max epochs: pretrain=1, finetune=1, supervised=1


In [98]:
# Weights & Biases configuration
WANDB_PROJECT = "HIPPIE"
WANDB_TAG = "cross_dataset_notebook"
UPLOAD_MODEL = False

RUN_NAME = f"{WANDB_TAG}-train_{TRAINING_DATASET}-predict_{PREDICT_DATASET}-{MODEL_CONFIG}_z{Z_DIM}_B{BETA}"

print(f"Project: {WANDB_PROJECT}")
print(f"Run name: {RUN_NAME}")

Project: HIPPIE
Run name: cross_dataset_notebook-train_hausser_cell_type-predict_mouse_organoids_cell_line-augmentation_ablation_z20_B0.9


## 3. Helper Functions

In [99]:
class ResourceMonitor(pl.Callback):
    """Logs GPU/CPU memory and average step time to W&B every N steps."""
    def __init__(self, log_every_n_steps: int = 50, namespace: str = "resources"):
        self.log_every_n_steps = max(1, log_every_n_steps)
        self.ns = namespace
        self._last_time = None
        self._accum_step_time = 0.0
        self._accum_steps = 0

    def on_train_start(self, trainer, pl_module):
        self._reset_cuda_peaks()
        self._last_time = time.perf_counter()

    def on_train_epoch_start(self, trainer, pl_module):
        self._reset_cuda_peaks()

    def on_train_batch_end(self, trainer, pl_module, outputs, batch, batch_idx):
        now = time.perf_counter()
        if self._last_time is not None:
            self._accum_step_time += (now - self._last_time)
            self._accum_steps += 1
        self._last_time = now

        global_step = trainer.global_step
        if global_step % self.log_every_n_steps == 0 and global_step > 0:
            metrics = {}
            if self._accum_steps > 0:
                metrics[f"{self.ns}/avg_step_time_s"] = self._accum_step_time / self._accum_steps
                self._accum_step_time, self._accum_steps = 0.0, 0

            if _HAS_PSUTIL:
                process = psutil.Process(os.getpid())
                rss_mb = process.memory_info().rss / (1024 ** 2)
                metrics[f"{self.ns}/cpu_rss_mb"] = rss_mb

            if torch.cuda.is_available():
                for d in range(torch.cuda.device_count()):
                    device = torch.device(f"cuda:{d}")
                    curr = torch.cuda.memory_allocated(device) / (1024 ** 2)
                    reserved = torch.cuda.memory_reserved(device) / (1024 ** 2)
                    peak = torch.cuda.max_memory_allocated(device) / (1024 ** 2)
                    metrics[f"{self.ns}/gpu{d}_mem_alloc_mb"] = curr
                    metrics[f"{self.ns}/gpu{d}_mem_reserved_mb"] = reserved
                    metrics[f"{self.ns}/gpu{d}_mem_peak_mb"] = peak

            if metrics:
                wandb.log(metrics, step=global_step)

    def on_validation_epoch_end(self, trainer, pl_module):
        if torch.cuda.is_available():
            metrics = {}
            for d in range(torch.cuda.device_count()):
                peak = torch.cuda.max_memory_allocated(d) / (1024 ** 2)
                metrics[f"{self.ns}/val_gpu{d}_mem_peak_mb"] = peak
            if metrics:
                wandb.log(metrics, step=trainer.global_step)
        self._reset_cuda_peaks()

    def _reset_cuda_peaks(self):
        if torch.cuda.is_available():
            for d in range(torch.cuda.device_count()):
                torch.cuda.reset_peak_memory_stats(d)

print("ResourceMonitor class defined")

ResourceMonitor class defined


In [100]:
def create_balanced_sampler(dataset, labels):
    unique_labels, label_counts = np.unique(labels, return_counts=True)
    class_weights = 1.0 / label_counts
    
    sample_weights = np.zeros(len(labels))
    for label_idx, label in enumerate(unique_labels):
        mask = labels == label
        sample_weights[mask] = class_weights[label_idx]
    
    sampler = WeightedRandomSampler(
        weights=torch.FloatTensor(sample_weights),
        num_samples=len(dataset),
        replacement=True
    )
    
    print(f"Class-Balanced Sampling Enabled")
    print(f"Number of classes: {len(unique_labels)}")
    for label, count in zip(unique_labels, label_counts):
        print(f"  Class {label}: {count} samples ({100*count/len(labels):.2f}%)")
    
    return sampler

def _log_timer(timer_obj: Timer, prefix: str):
    def _safe_elapsed(phase):
        try:
            val = timer_obj.time_elapsed(phase)
            return float(val) if val is not None else None
        except Exception:
            return None

    elapsed_fit = _safe_elapsed("fit")
    elapsed_train = _safe_elapsed("train")
    elapsed_validate = _safe_elapsed("validate")

    payload = {}
    if elapsed_fit is not None:
        payload[f"time/{prefix}_fit_s"] = elapsed_fit
    if elapsed_train is not None:
        payload[f"time/{prefix}_train_s"] = elapsed_train
    if elapsed_validate is not None:
        payload[f"time/{prefix}_val_s"] = elapsed_validate

    if payload:
        wandb.log(payload)

print("Helper functions defined")

Helper functions defined


In [101]:
def get_embeddings_multimodal(loader, model):
    model.eval()
    all_embeddings = []
    all_labels = []

    with torch.no_grad():
        for sample in loader:
            try:
                out = model(sample)
            except TypeError:
                out = model(*sample)
            embedding = out[0].detach().cpu().numpy()

            std = np.std(embedding, axis=1, keepdims=True)
            std[std == 0] = 1.0
            embedding = (embedding - np.mean(embedding, axis=1, keepdims=True)) / std
            all_embeddings.extend(embedding)

            label = sample[1]
            if getattr(label, "ndim", 1) == 2:
                cls_label, _ = label.unbind(1)
            else:
                cls_label = label
            all_labels.extend(cls_label.detach().cpu().numpy())

    return np.array(all_embeddings), np.array(all_labels)

def _nan_sanitize(x: np.ndarray, name: str, dataset: str):
    if np.isnan(x).any():
        print(f"NaN values detected in {dataset}/{name}, replacing with 0")
        x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
    return x

# def load_dataset_data(dataset_name, dataset_files):
#     wf = pd.read_csv(f"./datasets_hippie/{dataset_name}/waveforms.csv").to_numpy()
#     isi = pd.read_csv(f"./datasets_hippie/{dataset_name}/isi_dist.csv").to_numpy()
#     acg_path = f"./datasets_hippie/{dataset_name}/acg.csv"
#     acg = pd.read_csv(acg_path).to_numpy() if os.path.exists(acg_path) else np.zeros_like(isi)

#     wf = _nan_sanitize(wf, "wave", dataset_name)
#     isi = _nan_sanitize(isi, "isi", dataset_name)
#     acg = _nan_sanitize(acg, "acg", dataset_name)

#     labels = None
#     labels_path = f"./datasets_hippie/{dataset_name}/labels.csv"
#     if os.path.exists(labels_path):
#         labels_df = pd.read_csv(labels_path)
#         labels = labels_df[labels_df.columns[0]].values
#     elif os.path.exists(f"./datasets_hippie/{dataset_name}/celltypes.csv"):
#         labels_df = pd.read_csv(f"./datasets_hippie/{dataset_name}/celltypes.csv")
#         labels = labels_df[labels_df.columns[0]].values

#     source_id = dataset_files[dataset_name]
#     print(f"{dataset_name}: waveform={wf.shape}, isi={isi.shape}, acg={acg.shape}")
#     return wf, isi, acg, labels, source_id

def load_dataset_data(dataset_name, dataset_files):
    wf = pd.read_csv(f"./datasets_hippie/{dataset_name}/waveforms.csv").to_numpy()
    isi = pd.read_csv(f"./datasets_hippie/{dataset_name}/isi_dist.csv").to_numpy()
    acg_path = f"./datasets_hippie/{dataset_name}/acg.csv"
    acg = pd.read_csv(acg_path).to_numpy() if os.path.exists(acg_path) else np.zeros_like(isi)

    wf = _nan_sanitize(wf, "wave", dataset_name)
    isi = _nan_sanitize(isi, "isi", dataset_name)
    acg = _nan_sanitize(acg, "acg", dataset_name)

    labels = None
    labels_path = f"./datasets_hippie/{dataset_name}/labels.csv"  # FIXED: Changed from ../datasets_hippie
    if os.path.exists(labels_path):
        labels_df = pd.read_csv(labels_path)
        # FIXED: Fill NaN values with empty string before converting to numpy
        labels = labels_df[labels_df.columns[0]].fillna('').values
    elif os.path.exists(f"./datasets_hippie/{dataset_name}/celltypes.csv"):  # FIXED: Changed path
        labels_df = pd.read_csv(f"./datasets_hippie/{dataset_name}/celltypes.csv")
        # FIXED: Fill NaN values with empty string
        labels = labels_df[labels_df.columns[0]].fillna('').values

    source_id = dataset_files[dataset_name]
    print(f"{dataset_name}: waveform={wf.shape}, isi={isi.shape}, acg={acg.shape}")
    return wf, isi, acg, labels, source_id

def map_labels_to_training_encoder(le_train: LabelEncoder, labels: np.ndarray, fallback: int = 0):
    train_set = set(le_train.classes_)
    out = np.empty(labels.shape[0], dtype=int)
    for i, lbl in enumerate(labels):
        if lbl in train_set:
            out[i] = le_train.transform([lbl])[0]
        else:
            out[i] = fallback
    return out

print("Data loading functions defined")

Data loading functions defined


## 4. Initialize Experiment

In [102]:
torch.manual_seed(42)
np.random.seed(42)

wandb.init(
    project=WANDB_PROJECT,
    name=RUN_NAME,
    config={
        "training_dataset": TRAINING_DATASET,
        "predict_dataset": PREDICT_DATASET,
        "config": MODEL_CONFIG,
        "z_dim": Z_DIM,
        "beta": BETA,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "batch_size": BATCH_SIZE,
    }
)

accelerator = "gpu" if torch.cuda.is_available() else "cpu"
print(f"Wandb run: {wandb.run.name}")
print(f"Accelerator: {accelerator}")

Wandb run: cross_dataset_notebook-train_hausser_cell_type-predict_mouse_organoids_cell_line-augmentation_ablation_z20_B0.9
Accelerator: cpu


In [103]:
if TRAINING_DATASET not in AVAILABLE_DATASETS:
    raise ValueError(f"Training dataset not found")
if PREDICT_DATASET not in AVAILABLE_DATASETS:
    raise ValueError(f"Predict dataset not found")

pretrain_datasets = AVAILABLE_DATASETS.copy()
if TRAINING_DATASET in pretrain_datasets:
    pretrain_datasets.pop(TRAINING_DATASET)
if PREDICT_DATASET in pretrain_datasets:
    pretrain_datasets.pop(PREDICT_DATASET)

print(f"Training: {TRAINING_DATASET}")
print(f"Prediction: {PREDICT_DATASET}")
print(f"Pretraining: {list(pretrain_datasets.keys())}")

num_sources = max(AVAILABLE_DATASETS.values()) + 1

modalities = {"wave": 50, "isi": 100, "acg": 200}
modality_weights = {"wave": WAVE_WEIGHT, "isi": ISI_WEIGHT, "acg": ACG_WEIGHT}

EXPERIMENT_CONFIGS = {
    "baseline": ExperimentConfigs.baseline(),
    "with_source": ExperimentConfigs.with_source(),
    "with_class": ExperimentConfigs.with_class(),
    "with_both_embeddings": ExperimentConfigs.with_both_embeddings(),
    "with_batch_norm": ExperimentConfigs.with_batch_norm(),
    "full_model": ExperimentConfigs.full_model(),
    "no_fusion": ExperimentConfigs.no_fusion(),
    "with_light_augmentations": ExperimentConfigs.with_light_augmentations(),
    "with_heavy_augmentations": ExperimentConfigs.with_heavy_augmentations(),
    "augmentation_ablation": ExperimentConfigs.augmentation_ablation(),
}
config = EXPERIMENT_CONFIGS[MODEL_CONFIG]

print(f"Model configuration: {MODEL_CONFIG}")
print(f"  Use source embedding: {config.use_source_embedding}")
print(f"  Use class embedding: {config.use_class_embedding}")
print(f"  Use fusion encoder: {config.use_fusion_encoder}")

Training: hausser_cell_type
Prediction: mouse_organoids_cell_line
Pretraining: ['hull_cell_type', 'lissberger_labeled_cell_type']
Model configuration: augmentation_ablation
  Use source embedding: True
  Use class embedding: True
  Use fusion encoder: True


## Phase 1: Pretraining

In [104]:
print("PHASE 1: PRETRAINING")

datasets_multi = []
labels_list = []

for folder in pretrain_datasets:
    wf = pd.read_csv(f"./datasets_hippie/{folder}/waveforms.csv").to_numpy()
    isi = pd.read_csv(f"./datasets_hippie/{folder}/isi_dist.csv").to_numpy()
    acg_path = f"./datasets_hippie/{folder}/acg.csv"
    acg = pd.read_csv(acg_path).to_numpy() if os.path.exists(acg_path) else np.zeros_like(isi)
    
    wf = _nan_sanitize(wf, "wave", folder)
    isi = _nan_sanitize(isi, "isi", folder)
    acg = _nan_sanitize(acg, "acg", folder)

    source = np.full((wf.shape[0]), pretrain_datasets[folder])
    print(f"{folder}: waveform={wf.shape}, isi={isi.shape}, acg={acg.shape}")

    data_dict = {"wave": wf, "isi": isi, "acg": acg}
    dataset_multi = MultiModalEphysDataset(data_dict, source, mode="multi", modality_sizes=modalities)
    
    if config.use_augmentations and config.augment_pretraining:
        dataset_multi = AugmentedMultiModalEphysDataset(dataset_multi, config, phase="pretraining")
        print(f"  Augmentations enabled")
    
    datasets_multi.append(dataset_multi)
    labels_list.append(source)

if not datasets_multi:
    print("No datasets for pretraining")
    joint_model = None
    joint_path = None
else:
    print(f"Loaded {len(datasets_multi)} datasets for pretraining")

PHASE 1: PRETRAINING
hull_cell_type: waveform=(206, 75), isi=(206, 100), acg=(206, 201)
  Augmentations enabled
NaN values detected in lissberger_labeled_cell_type/wave, replacing with 0
lissberger_labeled_cell_type: waveform=(1152, 100), isi=(1152, 100), acg=(1152, 201)
  Augmentations enabled
Loaded 2 datasets for pretraining


In [105]:
if datasets_multi:
    all_multi_dataset = torch.utils.data.ConcatDataset(datasets_multi)
    
    indices = list(range(len(all_multi_dataset)))
    train_size = int(TRAIN_VAL_SPLIT * len(indices))
    train_indices, val_indices = random_split(indices, [train_size, len(indices) - train_size])
    
    train_multi_dataset = torch.utils.data.Subset(all_multi_dataset, train_indices)
    val_multi_dataset = torch.utils.data.Subset(all_multi_dataset, val_indices)
    
    train_loader_multi = torch.utils.data.DataLoader(
        train_multi_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=none_safe_collate
    )
    val_loader_multi = torch.utils.data.DataLoader(
        val_multi_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=none_safe_collate
    )
    
    print(f"Training: {len(train_multi_dataset)} samples")
    print(f"Validation: {len(val_multi_dataset)} samples")

Training: 1086 samples
Validation: 272 samples


In [106]:
if datasets_multi:
    joint_model = MultiModalCVAE(
        modalities=modalities,
        z_dim=Z_DIM,
        num_sources=num_sources,
        num_classes=5,
        config=config,
    )
    
    joint_model = MultiModalCVAETrainModule(
        joint_model,
        modality_weights=modality_weights,
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        config=config,
    )
    
    print(f"Model created with z_dim={Z_DIM}")

Model created with z_dim=20


In [107]:
if datasets_multi:
    checkpoint_callback = pl.callbacks.ModelCheckpoint(monitor="val_loss", save_top_k=1, mode="min")
    early_stop_callback = pl.callbacks.EarlyStopping(monitor="val_loss", patience=EARLY_STOPPING_PATIENCE, mode="min")
    timer_pretrain = Timer(duration=None)
    resource_monitor = ResourceMonitor(log_every_n_steps=50)
    
    wandb.log({"phase": "pretrain_start"})
    
    trainer = pl.Trainer(
        max_epochs=PRETRAIN_MAX_EPOCHS,
        accelerator=accelerator,
        logger=pl.loggers.WandbLogger(experiment=wandb.run),
        callbacks=[checkpoint_callback, early_stop_callback, timer_pretrain, resource_monitor],
        gradient_clip_val=GRADIENT_CLIP_VAL,
    )
    
    print(f"Starting pretraining (max {PRETRAIN_MAX_EPOCHS} epochs)")
    trainer.fit(joint_model, train_loader_multi, val_loader_multi)
    _log_timer(timer_pretrain, prefix="pretrain")
    
    joint_path = checkpoint_callback.best_model_path
    joint_model.load_state_dict(torch.load(joint_path)["state_dict"])
    print(f"Pretraining complete: {joint_path}")

GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs

  | Name     | Type           | Params | Mode 
----------------------------------------------------
0 | model    | MultiModalCVAE | 24.3 M | train
1 | mse_loss | MSELoss        | 0      | train
----------------------------------------------------
24.3 M    Trainable params
0         Non-trainable params
24.3 M    Total params
97.017    Total estimated model params size (MB)
426       Modules in train mode
0         Modules in eval mode


Starting pretraining (max 1 epochs)
Epoch 0: 100%|██████████| 3/3 [01:30<00:00,  0.03it/s, v_num=rhir]Average training loss is 2.54


`Trainer.fit` stopped: `max_epochs=1` reached.


Epoch 0: 100%|██████████| 3/3 [01:31<00:00,  0.03it/s, v_num=rhir]
Pretraining complete: ./lightning_logs/m4cmrhir/checkpoints/epoch=0-step=3.ckpt


## Phase 2: Fine-tuning

In [108]:
print("PHASE 2: FINE-TUNING")

train_wf, train_isi, train_acg, train_labels, train_source_id = load_dataset_data(TRAINING_DATASET, AVAILABLE_DATASETS)

if train_labels is None:
    raise ValueError(f"Training dataset must have labels")

print(f"Loaded {TRAINING_DATASET}: {len(train_wf)} samples")
unique_labels, counts = np.unique(train_labels, return_counts=True)
for label, count in zip(unique_labels, counts):
    print(f"  {label}: {count} ({100*count/len(train_labels):.1f}%)")

PHASE 2: FINE-TUNING
NaN values detected in hausser_cell_type/wave, replacing with 0
hausser_cell_type: waveform=(3996, 75), isi=(3996, 100), acg=(3996, 201)
Loaded hausser_cell_type: 3996 samples
  b'': 3770 (94.3%)
  b'GoC': 50 (1.3%)
  b'MFB': 26 (0.7%)
  b'MLI': 30 (0.8%)
  b'PkC_cs': 50 (1.3%)
  b'PkC_ss': 70 (1.8%)


In [109]:
if joint_model is not None and FINETUNE_WITHOUT_LABELS:
    finetune_data_dict = {"wave": train_wf, "isi": train_isi, "acg": train_acg}
    label_ft = np.full((train_wf.shape[0]), train_source_id)
    finetune_dataset_multi = MultiModalEphysDataset(finetune_data_dict, label_ft, mode="multi", modality_sizes=modalities)
    
    indices = list(range(len(finetune_dataset_multi)))
    train_size = int(FINETUNE_SPLIT * len(indices))
    train_indices, val_indices = random_split(indices, [train_size, len(indices) - train_size])
    
    original_model = joint_model.model
    joint_model = MultiModalCVAETrainModule(
        original_model,
        modality_weights=modality_weights,
        learning_rate=LEARNING_RATE / 10,
        weight_decay=WEIGHT_DECAY,
        config=config,
    )
    
    if config.use_augmentations and config.augment_finetuning:
        augmented_finetune_dataset = AugmentedMultiModalEphysDataset(finetune_dataset_multi, config, phase="finetuning")
        train_finetune_dataset = torch.utils.data.Subset(augmented_finetune_dataset, train_indices)
        print("Augmentations enabled")
    else:
        train_finetune_dataset = torch.utils.data.Subset(finetune_dataset_multi, train_indices)
    
    val_finetune_dataset = torch.utils.data.Subset(finetune_dataset_multi, val_indices)
    
    train_finetune_loader = torch.utils.data.DataLoader(
        train_finetune_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=none_safe_collate
    )
    val_finetune_loader = torch.utils.data.DataLoader(
        val_finetune_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=none_safe_collate
    )
    
    print(f"Training: {len(train_finetune_dataset)} samples")
    print(f"Validation: {len(val_finetune_dataset)} samples")

Augmentations enabled
Training: 799 samples
Validation: 3197 samples


In [110]:
if joint_model is not None and FINETUNE_WITHOUT_LABELS:
    checkpoint_callback = pl.callbacks.ModelCheckpoint(monitor="val_loss", save_top_k=1, mode="min")
    early_stop_callback = pl.callbacks.EarlyStopping(monitor="val_loss", patience=EARLY_STOPPING_PATIENCE, mode="min")
    timer_finetune = Timer(duration=None)
    resource_monitor = ResourceMonitor(log_every_n_steps=50)
    
    wandb.log({"phase": "finetune_start"})
    
    trainer = pl.Trainer(
        max_epochs=FINETUNE_MAX_EPOCHS,
        accelerator=accelerator,
        logger=pl.loggers.WandbLogger(experiment=wandb.run),
        callbacks=[checkpoint_callback, early_stop_callback, timer_finetune, resource_monitor],
        gradient_clip_val=GRADIENT_CLIP_VAL,
    )
    
    print(f"Starting fine-tuning (max {FINETUNE_MAX_EPOCHS} epochs)")
    trainer.fit(joint_model, train_finetune_loader, val_finetune_loader)
    _log_timer(timer_finetune, prefix="finetune")
    
    joint_path = checkpoint_callback.best_model_path
    joint_model.load_state_dict(torch.load(joint_path)["state_dict"])
    print(f"Fine-tuning complete: {joint_path}")
else:
    print("Skipping fine-tuning")

GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs

  | Name     | Type           | Params | Mode 
----------------------------------------------------
0 | model    | MultiModalCVAE | 24.3 M | train
1 | mse_loss | MSELoss        | 0      | train
----------------------------------------------------
24.3 M    Trainable params
0         Non-trainable params
24.3 M    Total params
97.017    Total estimated model params size (MB)
426       Modules in train mode
0         Modules in eval mode


Starting fine-tuning (max 1 epochs)
Epoch 0: 100%|██████████| 2/2 [02:08<00:00,  0.02it/s, v_num=rhir]Average training loss is 1.59


`Trainer.fit` stopped: `max_epochs=1` reached.


Epoch 0: 100%|██████████| 2/2 [02:08<00:00,  0.02it/s, v_num=rhir]
Fine-tuning complete: ./lightning_logs/m4cmrhir/checkpoints/epoch=0-step=2.ckpt


## Phase 3: Supervised Training

In [111]:
print("PHASE 3: SUPERVISED TRAINING")

le = LabelEncoder().fit(train_labels)
train_labels_encoded = le.transform(train_labels)

print("Label encoding:")
for i, label in enumerate(le.classes_):
    print(f"  {i}: {label}")

indices = list(range(len(train_wf)))
train_size = int(TRAIN_VAL_SPLIT * len(indices))
train_indices, val_indices = random_split(indices, [train_size, len(indices) - train_size])

wf_train = train_wf[train_indices]
wf_val = train_wf[val_indices]
isi_train = train_isi[train_indices]
isi_val = train_isi[val_indices]
acg_train = train_acg[train_indices]
acg_val = train_acg[val_indices]
label_train = train_labels_encoded[train_indices]
label_val = train_labels_encoded[val_indices]

num_class_labels = len(np.unique(label_train))

print(f"Training: {len(wf_train)} samples")
print(f"Validation: {len(wf_val)} samples")
print(f"Classes: {num_class_labels}")

PHASE 3: SUPERVISED TRAINING
Label encoding:
  0: b''
  1: b'GoC'
  2: b'MFB'
  3: b'MLI'
  4: b'PkC_cs'
  5: b'PkC_ss'
Training: 3196 samples
Validation: 800 samples
Classes: 6


In [112]:
supervised_joint_model = MultiModalCVAE(
    modalities=modalities,
    z_dim=Z_DIM,
    num_sources=num_sources,
    num_classes=num_class_labels,
    config=config,
)

if joint_model is not None and joint_path is not None:
    joint_seq = torch.load(joint_path)
    if "model.class_embedding.weight" in joint_seq["state_dict"]:
        joint_seq["state_dict"].pop("model.class_embedding.weight")
    
    supervised_joint_model = MultiModalCVAETrainModule(
        supervised_joint_model,
        modality_weights=modality_weights,
        learning_rate=LEARNING_RATE / 10,
        weight_decay=WEIGHT_DECAY,
        config=config,
    )
    supervised_joint_model.load_state_dict(joint_seq["state_dict"], strict=False)
    print("Loaded pretrained weights")
else:
    supervised_joint_model = MultiModalCVAETrainModule(
        supervised_joint_model,
        modality_weights=modality_weights,
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        config=config,
    )
    print("Starting from scratch")

Loaded pretrained weights


In [113]:
label_train_for_embedding = train_source_id * np.ones_like(label_train)
label_val_for_embedding = train_source_id * np.ones_like(label_val)

train_data_dict = {"wave": wf_train, "isi": isi_train, "acg": acg_train}
val_data_dict = {"wave": wf_val, "isi": isi_val, "acg": acg_val}

dataset_train_multi = MultiModalEphysDataset(
    train_data_dict,
    np.vstack((label_train, label_train_for_embedding)).T,
    mode="multi",
    modality_sizes=modalities
)

if config.use_augmentations and config.augment_supervised:
    dataset_train_multi = AugmentedMultiModalEphysDataset(dataset_train_multi, config, phase="supervised")
    print("Augmentations enabled")

dataset_val_multi = MultiModalEphysDataset(
    val_data_dict,
    np.vstack((label_val, label_val_for_embedding)).T,
    mode="multi",
    modality_sizes=modalities
)

if USE_BALANCED_SAMPLING:
    train_sampler = create_balanced_sampler(dataset_train_multi, label_train)
    train_loader_multi = torch.utils.data.DataLoader(
        dataset_train_multi,
        batch_size=SUPERVISED_BATCH_SIZE,
        sampler=train_sampler,
        collate_fn=none_safe_collate,
    )
else:
    train_loader_multi = torch.utils.data.DataLoader(
        dataset_train_multi,
        batch_size=SUPERVISED_BATCH_SIZE,
        shuffle=True,
        collate_fn=none_safe_collate,
    )

val_loader_multi = torch.utils.data.DataLoader(
    dataset_val_multi,
    batch_size=SUPERVISED_BATCH_SIZE,
    shuffle=False,
    collate_fn=none_safe_collate
)

print(f"Dataloaders created")

Class-Balanced Sampling Enabled
Number of classes: 6
  Class 0: 3016 samples (94.37%)
  Class 1: 38 samples (1.19%)
  Class 2: 22 samples (0.69%)
  Class 3: 25 samples (0.78%)
  Class 4: 38 samples (1.19%)
  Class 5: 57 samples (1.78%)
Dataloaders created


In [114]:
checkpoint_callback = pl.callbacks.ModelCheckpoint(monitor="val_loss", save_top_k=1, mode="min")
early_stop_callback = pl.callbacks.EarlyStopping(monitor="val_loss", patience=EARLY_STOPPING_PATIENCE, mode="min")
lr_monitor = pl.callbacks.LearningRateMonitor(logging_interval="step")
timer_supervised = Timer(duration=None)
resource_monitor = ResourceMonitor(log_every_n_steps=50)

wandb.log({"phase": "supervised_start"})

trainer = pl.Trainer(
    max_epochs=SUPERVISED_MAX_EPOCHS,
    accelerator=accelerator,
    logger=pl.loggers.WandbLogger(experiment=wandb.run),
    callbacks=[checkpoint_callback, early_stop_callback, lr_monitor, timer_supervised, resource_monitor],
    gradient_clip_val=GRADIENT_CLIP_VAL,
)

print(f"Starting supervised training (max {SUPERVISED_MAX_EPOCHS} epochs)")
trainer.fit(supervised_joint_model, train_loader_multi, val_loader_multi)
_log_timer(timer_supervised, prefix="supervised")

joint_path = checkpoint_callback.best_model_path
wandb.log({"best_epoch_joint": joint_path})

joint_seq = torch.load(joint_path)
supervised_joint_model.load_state_dict(joint_seq["state_dict"])
supervised_joint_model.eval()

print(f"Supervised training complete: {joint_path}")

GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs

  | Name     | Type           | Params | Mode 
----------------------------------------------------
0 | model    | MultiModalCVAE | 24.3 M | train
1 | mse_loss | MSELoss        | 0      | train
----------------------------------------------------
24.3 M    Trainable params
0         Non-trainable params
24.3 M    Total params
97.017    Total estimated model params size (MB)
426       Modules in train mode
0         Modules in eval mode


Starting supervised training (max 1 epochs)
Epoch 0: 100%|██████████| 50/50 [04:54<00:00,  0.17it/s, v_num=rhir]Average training loss is 1.14


`Trainer.fit` stopped: `max_epochs=1` reached.


Epoch 0: 100%|██████████| 50/50 [04:55<00:00,  0.17it/s, v_num=rhir]
Supervised training complete: ./lightning_logs/m4cmrhir/checkpoints/epoch=0-step=50.ckpt


## Phase 4: Evaluation

In [115]:
print("PHASE 4: EVALUATION")
print("Extracting training embeddings")

train_full_data_dict = {"wave": train_wf, "isi": train_isi, "acg": train_acg}

train_full_dataset = MultiModalEphysDataset(
    train_full_data_dict,
    np.vstack((train_labels_encoded, np.ones_like(train_labels_encoded) * train_source_id)).T,
    mode="multi",
    modality_sizes=modalities
)

train_full_loader = torch.utils.data.DataLoader(train_full_dataset, batch_size=128, collate_fn=none_safe_collate)
train_embeddings, train_labels_final = get_embeddings_multimodal(train_full_loader, supervised_joint_model)

print(f"Extracted {len(train_embeddings)} training embeddings")

PHASE 4: EVALUATION
Extracting training embeddings
Extracted 3996 training embeddings


In [116]:
print("Extracting prediction embeddings")

predict_wf, predict_isi, predict_acg, predict_labels, predict_source_id = load_dataset_data(PREDICT_DATASET, AVAILABLE_DATASETS)

if predict_labels is None:
    print("No labels, creating dummy labels")
    predict_labels = np.zeros(len(predict_wf), dtype=str)
    predict_labels[:] = "unknown"

predict_labels_for_model = map_labels_to_training_encoder(le, predict_labels, fallback=0)

predict_data_dict = {"wave": predict_wf, "isi": predict_isi, "acg": predict_acg}

predict_dataset = MultiModalEphysDataset(
    predict_data_dict,
    np.vstack((predict_labels_for_model, np.ones_like(predict_labels_for_model) * predict_source_id)).T,
    mode="multi",
    modality_sizes=modalities
)

predict_loader = torch.utils.data.DataLoader(predict_dataset, batch_size=128, collate_fn=none_safe_collate)
predict_embeddings, predict_labels_final = get_embeddings_multimodal(predict_loader, supervised_joint_model)

print(f"Extracted {len(predict_embeddings)} prediction embeddings")

Extracting prediction embeddings
NaN values detected in mouse_organoids_cell_line/wave, replacing with 0
mouse_organoids_cell_line: waveform=(4745, 51), isi=(4745, 101), acg=(4745, 202)
Extracted 4745 prediction embeddings


In [117]:
print("Training KNN classifier")

train_knn_labels = train_labels_final.astype(int)
neighbor_options = list(range(5, min(20, len(np.unique(train_knn_labels)) * 3)))
if not neighbor_options:
    neighbor_options = [3, 5]

print(f"Training samples: {len(train_embeddings)}")
print(f"Testing k values: {neighbor_options}")

cv_scores = {}
for neighbor in neighbor_options:
    knn = KNeighborsClassifier(n_neighbors=neighbor)
    cv_score = cross_val_score(
        knn, train_embeddings, train_knn_labels,
        cv=min(5, len(np.unique(train_knn_labels))),
        scoring='balanced_accuracy'
    )
    cv_scores[neighbor] = np.mean(cv_score)
    print(f"  k={neighbor}: {np.mean(cv_score):.4f}")

best_neighbors = max(cv_scores, key=cv_scores.get)
print(f"Selected k={best_neighbors}")

final_knn = KNeighborsClassifier(n_neighbors=best_neighbors)
final_knn.fit(train_embeddings, train_knn_labels)

Training KNN classifier
Training samples: 3996
Testing k values: [5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17]
  k=5: 0.4884
  k=6: 0.4249
  k=7: 0.4261
  k=8: 0.3954
  k=9: 0.4100
  k=10: 0.3761
  k=11: 0.3855
  k=12: 0.3557
  k=13: 0.3604
  k=14: 0.3504
  k=15: 0.3632
  k=16: 0.3551
  k=17: 0.3551
Selected k=5


KNeighborsClassifier()

In [118]:
print("Making predictions")

predictions = final_knn.predict(predict_embeddings)
prediction_probabilities = final_knn.predict_proba(predict_embeddings)
predicted_labels = le.inverse_transform(predictions.astype(int))

print(f"Made {len(predictions)} predictions")

Making predictions
Made 4745 predictions


In [119]:
accuracy_calculated = False

if not (predict_labels == "unknown").all():
    train_classes = set(le.classes_)
    predict_classes = set(np.unique(predict_labels))
    overlapping_classes = train_classes.intersection(predict_classes)
    
    if overlapping_classes:
        print(f"Overlapping classes: {overlapping_classes}")
        
        true_labels_for_eval = []
        pred_labels_for_eval = []
        
        for i, (true_label, pred_idx) in enumerate(zip(predict_labels, predictions)):
            if true_label in overlapping_classes:
                true_labels_for_eval.append(le.transform([true_label])[0])
                pred_labels_for_eval.append(pred_idx)
        
        if len(true_labels_for_eval) > 0:
            accuracy = balanced_accuracy_score(true_labels_for_eval, pred_labels_for_eval)
            accuracy_calculated = True
            
            print(f"Balanced Accuracy: {accuracy:.4f}")
            print(f"Best k: {best_neighbors}")
            print(f"Overlapping classes: {len(overlapping_classes)}")
            print(f"Evaluated samples: {len(true_labels_for_eval)}")
            
            conf_matrix = confusion_matrix(true_labels_for_eval, pred_labels_for_eval)
            available_classes = [c for c in le.classes_ if c in overlapping_classes]
            
            wandb.log({
                "cross_dataset_balanced_accuracy": accuracy,
                "best_k_neighbors": best_neighbors,
                "num_overlapping_classes": len(overlapping_classes),
                "num_evaluated_samples": len(true_labels_for_eval)
            })
            
            try:
                figure_multi = make_confmat(conf_matrix, available_classes, best_neighbors)
                wandb.log({"cross_dataset_confusion_matrix": wandb.Image(figure_multi)})
                print("Confusion matrix logged")
            except Exception as e:
                print(f"Could not create confusion matrix: {e}")

if not accuracy_calculated:
    wandb.log({"best_k_neighbors": best_neighbors})

In [120]:
wandb.log({"phase": "complete"})

print("CROSS-DATASET TRAINING COMPLETE")
print(f"Configuration: {MODEL_CONFIG}")
print(f"Training dataset: {TRAINING_DATASET}")
print(f"Prediction dataset: {PREDICT_DATASET}")
print(f"Training classes: {list(le.classes_)}")
print(f"Best k: {best_neighbors}")

# Create output directory
output_dir = f"./outputs/{wandb.run.name}"
os.makedirs(output_dir, exist_ok=True)

if accuracy_calculated:
    print(f"Accuracy: {accuracy:.4f}")

# Save training embeddings with true labels
print("Saving training embeddings...")
train_embedding_cols = [f"embedding_{i}" for i in range(train_embeddings.shape[1])]
train_df = pd.DataFrame(train_embeddings, columns=train_embedding_cols)
train_df['true_label'] = train_labels  # Original string labels
train_csv_path = os.path.join(output_dir, "train_embeddings.csv")
train_df.to_csv(train_csv_path, index=False)
print(f"  Saved {len(train_df)} training embeddings to {train_csv_path}")

# Save prediction embeddings with predicted and true labels
print("Saving prediction embeddings...")
pred_embedding_cols = [f"embedding_{i}" for i in range(predict_embeddings.shape[1])]
pred_df = pd.DataFrame(predict_embeddings, columns=pred_embedding_cols)
pred_df['predicted_label'] = predicted_labels  # Predicted string labels
pred_df['true_label'] = predict_labels  # Original true labels (or "unknown")

# Add prediction probabilities for each class
for i, class_name in enumerate(le.classes_):
    pred_df[f'prob_{class_name}'] = prediction_probabilities[:, i]

pred_csv_path = os.path.join(output_dir, "predict_embeddings.csv")
pred_df.to_csv(pred_csv_path, index=False)
print(f"  Saved {len(pred_df)} prediction embeddings to {pred_csv_path}")

print(f"\nAll outputs saved to: {output_dir}")
print(f"  - train_embeddings.csv: {train_df.shape}")
print(f"  - predict_embeddings.csv: {pred_df.shape}")

wandb.finish()
print("\nExperiment complete")

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


CROSS-DATASET TRAINING COMPLETE
Configuration: augmentation_ablation
Training dataset: hausser_cell_type
Prediction dataset: mouse_organoids_cell_line
Training classes: ["b''", "b'GoC'", "b'MFB'", "b'MLI'", "b'PkC_cs'", "b'PkC_ss'"]
Best k: 5
Saving training embeddings...
  Saved 3996 training embeddings to ./outputs/cross_dataset_notebook-train_hausser_cell_type-predict_mouse_organoids_cell_line-augmentation_ablation_z20_B0.9/train_embeddings.csv
Saving prediction embeddings...
  Saved 4745 prediction embeddings to ./outputs/cross_dataset_notebook-train_hausser_cell_type-predict_mouse_organoids_cell_line-augmentation_ablation_z20_B0.9/predict_embeddings.csv

All outputs saved to: ./outputs/cross_dataset_notebook-train_hausser_cell_type-predict_mouse_organoids_cell_line-augmentation_ablation_z20_B0.9
  - train_embeddings.csv: (3996, 21)
  - predict_embeddings.csv: (4745, 28)


best_k_neighbors,▁
embedding_warmup_factor,▁
epoch,▁▁▁▁
lr-AdamW,▁
resources/avg_step_time_s,▁
resources/cpu_rss_mb,▁
time/finetune_train_s,▁
time/finetune_val_s,▁
time/pretrain_train_s,▁
time/pretrain_val_s,▁
+14,...



Experiment complete
